# Campina Grande (PB) · Acesso às UBS
Quantos moradores vivem em setores a mais de 2 km de uma UBS? Execute as três etapas em ordem. Usamos apenas os setores de Campina Grande (IBGE, 2022) e as UBS do cadastro de 19/11/2026. Os 2 km são um raio experimental em linha reta.

## 1. Carregar os dados do município

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist

base = 'https://raw.githubusercontent.com/L42Matheus/colab-ubs/1fbaedae2155509bd7ac07bbcf21351e2a573e3a/dados/'
setores = gpd.read_file(base + 'setores.geojson').sort_values('CD_SETOR').to_crs(31985)
cadastro = pd.read_csv(base + 'ubs.csv', dtype={'CNES': str})
coordenadas = cadastro.dropna(subset=['LONGITUDE', 'LATITUDE'])
ubs = gpd.GeoDataFrame(coordenadas, geometry=gpd.points_from_xy(
    coordenadas.LONGITUDE, coordenadas.LATITUDE), crs=4326).to_crs(31985)
ubs = ubs[ubs.geometry.intersects(setores.geometry.union_all())].drop_duplicates('CNES')
print(f'Campina Grande: {len(setores)} setores e {len(ubs)} UBS com coordenadas no município.')

## 2. Mapear os setores a mais de 2 km
Representamos cada setor por um ponto interno. O mapa destaca os setores cujo ponto fica além do raio da UBS mais próxima.

In [ ]:
pontos = setores.geometry.representative_point()
xy = np.column_stack([pontos.x, pontos.y])
xy_ubs = np.column_stack([ubs.geometry.x, ubs.geometry.y])
raio = 2000
distancias = cdist(xy, xy_ubs)
fora = distancias.min(axis=1) > raio
populacao = setores.populacao.to_numpy(dtype=int)

ax = setores.plot(color='#ececec', edgecolor='#aaaaaa', linewidth=0.15, figsize=(9, 7))
setores.loc[fora].plot(ax=ax, color='#e49253')
ubs.plot(ax=ax, color='#114f63', markersize=9, label='UBS cadastrada')
ax.set_title('Campina Grande · setores além de 2 km em laranja')
ax.legend()
ax.set_axis_off()
plt.show()
print(f'{fora.sum()} setores além do raio, com {populacao[fora].sum()} moradores estimados.')

## 3. Simular até três novos pontos
Em cada rodada, escolhemos o ponto de setor que alcança mais moradores ainda fora do raio. As coordenadas indicam áreas para avaliação em campo.

In [ ]:
vizinhos = cdist(xy, xy) <= raio
pendentes = fora.copy()
locais = pontos.to_crs(4326)
linhas = [[0, 'Rede cadastrada', None, None, 0, populacao[pendentes].sum()]]
for rodada in range(1, 4):
    ganhos = vizinhos @ (populacao * pendentes)
    candidato = ganhos.argmax()
    if ganhos[candidato] == 0:
        break
    pendentes &= ~vizinhos[candidato]
    local = locais.iloc[candidato]
    linhas.append([rodada, setores.iloc[candidato].CD_SETOR, local.y, local.x,
                   ganhos[candidato], populacao[pendentes].sum()])

pd.DataFrame(linhas, columns=['Novos pontos', 'Setor', 'Latitude', 'Longitude',
                              'Moradores alcançados na rodada', 'Moradores além de 2 km'])